# 03 - Two-Period DiD: NYC

Estimate the ATT of Local Law 18 using a simple two-period DiD. NYC vs control cities, pre vs post September 2023.

In [ ]:
# Data setup
# Set DATA_FILE to 'city_month_panel_real.parquet' after running build_real_panel.py
DATA_FILE = "city_month_panel_real.parquet"   # real Inside Airbnb data
# DATA_FILE = "city_month_panel.parquet"       # synthetic fallback (not committed)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

panel = pd.read_parquet(DATA_DIR / DATA_FILE)
panel["month"] = pd.to_datetime(panel["month"])

regs = pd.read_csv("../data/regulations.csv", parse_dates=["enforcement_date"])

print(f"Panel: {panel.shape}  |  Cities: {sorted(panel['city'].unique())}")
print(f"Date range: {panel['month'].min().date()} to {panel['month'].max().date()}")

In [ ]:
import statsmodels.formula.api as smf

## Build two-period dataset

In [ ]:
NYC_TREAT = pd.Timestamp("2023-09-01")
panel["treat_nyc"] = (panel["city"] == "New York City").astype(int)
panel["post_nyc"]  = (panel["month"] >= NYC_TREAT).astype(int)
panel["did_nyc"]   = panel["treat_nyc"] * panel["post_nyc"]

# Exclude Florence (also treated) from the two-period DiD control group
did_data = panel[panel["city"] != "Florence"].copy()
print(f"DiD sample: {did_data['city'].unique()}")

## Model 1 - Basic 2x2 DiD

In [ ]:
m1 = smf.ols(
    "log_listings ~ treat_nyc + post_nyc + did_nyc",
    data=did_data
).fit(cov_type="HC3")

att = m1.params["did_nyc"]
ci  = m1.conf_int().loc["did_nyc"]
print(f"ATT (log_listings) = {att:.4f}  95% CI [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"In levels: {(np.exp(att)-1)*100:+.1f}% change in listings")

## Model 2 - TWFE with city and month FE (preferred)

In [ ]:
from linearmodels.panel import PanelOLS

did_idx = did_data.set_index(["city", "month"])
fe = PanelOLS(
    did_idx["log_listings"],
    did_idx[["did_nyc"]],
    entity_effects=True,
    time_effects=True,
).fit(cov_type="clustered", cluster_entity=True)

att_fe = fe.params["did_nyc"]
ci_fe  = fe.conf_int().loc["did_nyc"]
print(f"TWFE ATT (log_listings) = {att_fe:.4f}  95% CI [{ci_fe['lower']:.4f}, {ci_fe['upper']:.4f}]")
print(f"In levels: {(np.exp(att_fe)-1)*100:+.1f}% change in listings")
print()
print(fe.summary.tables[1])

## All outcomes

In [ ]:
results = {}
for outcome in ["log_listings","log_price","availability_rate","entire_home_share"]:
    fe_m = PanelOLS(
        did_idx[outcome],
        did_idx[["did_nyc"]],
        entity_effects=True, time_effects=True,
    ).fit(cov_type="clustered", cluster_entity=True)
    b  = fe_m.params["did_nyc"]
    ci = fe_m.conf_int().loc["did_nyc"]
    results[outcome] = (b, ci["lower"], ci["upper"])
    print(f"{outcome:<22}  ATT = {b:+.4f}  [{ci['lower']:+.4f}, {ci['upper']:+.4f}]")